In [1]:
import sqlite3
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
import IPython.display as display

In [2]:
def load_data():
    conn = sqlite3.connect('THEA.db')
    
    df_eye = pd.read_sql("SELECT * FROM eyemate_measurements", conn)
    df_visits = pd.read_sql("SELECT * FROM all_visits", conn)

    conn.close()
    return df_eye, df_visits

df_eye, df_visits = load_data()

In [3]:
patient_list = list(set(df_eye['patient_id'].unique()) | set(df_visits['patient_id'].unique()))

In [4]:
@widgets.interact(
    patient_id = widgets.Dropdown(options=patient_list, description="Patient ID:"),
    alpha = widgets.FloatSlider(value=0.1, min=0.01, max=1, step=0.1, description="EWMA Alpha:")
)

def render_ewma(patient_id, alpha):
    fig = go.Figure()

    pdf_eye = df_eye[df_eye['patient_id'] == patient_id].copy()
    pdf_visits = df_visits[df_visits['patient_id'] == patient_id].copy()

    if not pdf_eye.empty:
        pdf_eye = pdf_eye.sort_values('time_of_measurement')
        pdf_eye['ewma'] = pdf_eye['eye_pressure'].ewm(alpha=alpha, adjust=False).mean()
    
        fig.add_trace(go.Scatter(
            x=pdf_eye['time_of_measurement'],
            y=pdf_eye['eye_pressure'],
            mode='markers',
            marker=dict(color='rgba(59, 130, 246, 0.4)', size=6),
            name='Raw IOP'
        ))

        fig.add_trace(go.Scatter(
            x=pdf_eye['time_of_measurement'],
            y=pdf_eye['ewma'],
            mode='lines',
            line=dict(shape='spline', smoothing=0.5, color="#1e750d", width=2.5)
        ))

    fig.add_hline(y=21, line_dash="dash", line_color="rgba(239, 68,68, 0.6)", annotation_text="Limit: 21mmHg")

    fig.update_layout(
        title=f"IOP EWMA - Patient {patient_id}",
        yaxis_title="IOP (mmHg)",
        hovermode="x unified",
        template="plotly_white",
        margin=dict(l=20, r=20, t=50, b=20),
        height=500
    )

    fig.show()

interactive(children=(Dropdown(description='Patient ID:', options=('DE-SC01-3-04', '02-09-001', 'DE-SC01-3-02'…

In [10]:
patient_list = df_eye[df_eye['patient_id'].str.contains("DE-SC01", na=False)]['patient_id'].unique().tolist()

@widgets.interact(
    patient_id=widgets.Dropdown(options=patient_list, description='Patient ID:'),
    window_days=widgets.IntSlider(value=7, min=1, max=60, step=1, description="Window (Days):")
)

def render_boxplot(patient_id, window_days):
    pdf = df_eye[df_eye['patient_id'] == patient_id].copy()
    pdf_visits = df_visits[df_visits['patient_id'] == patient_id].copy()
    
    pdf['time_of_measurement'] = pd.to_datetime(pdf['time_of_measurement'])

    start_date = pdf['time_of_measurement'].min().floor('D')
    print(start_date)
    pdf['days_since'] = (pdf['time_of_measurement'] - start_date).dt.days
    pdf['bin_idx'] = pdf['days_since'] // window_days
    
    # Map the bin index back to a readable starting date for the X-axis
    pdf['Window_Start'] = start_date + pd.to_timedelta(pdf['bin_idx'] * window_days, unit='D')
    
    # FIX 1: We no longer need the string 'Window_Label'. 
    # We will use the actual 'Window_Start' datetime objects for everything.
    
    pdf = pdf.sort_values('Window_Start')

    # FIX 2: Group by the datetime object instead of the string label
    df_medians = pdf.groupby('Window_Start', sort=False)['eye_pressure'].median().reset_index()

    # --- Create the Boxplot ---
    fig = go.Figure()
    
    fig.add_trace(go.Box(
        x=pdf['Window_Start'], # FIX 3: Pass datetime objects to the Boxplot
        y=pdf['eye_pressure'],
        name='Eye Pressure',
        marker_color='#3b82f6',
        boxmean='sd',
        boxpoints='outliers' 
    ))

    fig.add_trace(go.Scatter(
        x=df_medians['Window_Start'], # FIX 4: Pass datetime objects here too
        y=df_medians['eye_pressure'],
        mode='lines',
        name="Median Trends",
        line=dict(color='#cf6b3a', width=3)
    ))

    if not pdf_visits.empty:
        pdf_visits['date'] = pd.to_datetime(pdf_visits['date'], errors="coerce")

        for col in ['gat_mean', 'argos_mean']:
            if col in pdf_visits.columns:
                if pdf_visits[col].dtype == "object":
                    pdf_visits[col] = pdf_visits[col].astype(str).str.replace(',', '.')
                pdf_visits[col] = pd.to_numeric(pdf_visits[col], errors="coerce")

        pdf_visits_sorted = pdf_visits.sort_values('date')

        # GAT
        fig.add_trace(go.Scatter(
            x=pdf_visits_sorted['date'], # This is a datetime, which now matches the rest of the plot!
            y=pdf_visits_sorted['gat_mean'],
            mode="lines+markers",
            name="GAT Mean",
            marker=dict(symbol='square', size=8),
            line=dict(color="#cf6b3a", width=2, dash='dot'),
            connectgaps=True
        ))

        # ARGOS Line
        fig.add_trace(go.Scatter(
            x=pdf_visits_sorted['date'], 
            y=pdf_visits_sorted['argos_mean'], 
            mode='lines+markers', 
            name='ARGOS Mean',
            marker=dict(symbol='circle', size=8), 
            line=dict(color='#8b5cf6', width=2, dash='dot'),
            connectgaps=True
        ))

    # --- Layout Updates ---
    fig.update_layout(
        title=f"IOP Distribution (Grouped every {window_days} days) - Patient {patient_id}",
        xaxis_title="Window Start Date",
        yaxis_title="Intraocular Pressure (mmHg)",
        template="plotly_white",
        margin=dict(l=20, r=20, t=50, b=20),
        height=700
        # FIX 5: I removed xaxis=dict(type='category') so Plotly scales the timeline accurately
    )
    
    # Add the clinical target line for reference
    fig.add_hline(y=21, line_dash="dash", line_color="rgba(239, 68, 68, 0.6)", annotation_text="Limit: 21 mmHg")
    
    fig.show()

interactive(children=(Dropdown(description='Patient ID:', options=('DE-SC01-3-04', 'DE-SC01-3-02', 'DE-SC01-1-…